# Support Ticket Classification: Feature Engineering + MLP, BERT and GPT

Complete these activities in order:

- Inspect the shared ticket data and train/validation/test splits.
- Train a word-count MLP, fine-tune DistilBERT and fine-tune GPT-2.
- Compare the three methods, inspect errors and save models for new tickets.

[Open in Colab](https://colab.research.google.com/github/wlos/AI_SS/blob/main/14-15-Sequence-Models-and-Transformers/practice/Session14-15%20Support%20Ticket%20Classification%20Practice.ipynb)

Select a GPU runtime for Transformer fine-tuning, then run the notebook from top to bottom. The first code cell installs the required packages. The data cell downloads the [ticket CSV](https://raw.githubusercontent.com/wlos/AI_SS/main/14-15-Sequence-Models-and-Transformers/sample_data/support_tickets.csv) into `sample_data/`, refreshing it when rerun. Model files are written to `transformer/` in the current working directory.

In Colab, download any saved models you want to keep before the runtime is deleted.


In [ ]:
import subprocess
import sys

# Install the libraries before importing them in later cells.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "torch>=2.6,<3", "transformers==4.57.1", "datasets>=3,<5", "accelerate>=1,<2",
    "pandas", "scikit-learn", "matplotlib", "joblib",
])


## 1. Data: Support Tickets and Shared Splits

Use the supplied **320 fictional tickets**: 208 for training, 48 for validation and 64 for testing. Each ticket has one of four labels: Account access, Hardware, Software or Network.

Run the next cell to refresh the CSV in `sample_data/`, inspect the data, and create the shared splits and label mapping.


In [ ]:
import os
from urllib.request import urlretrieve
import pandas as pd

TICKET_CSV = "sample_data/support_tickets.csv"
TICKET_CSV_URL = "https://raw.githubusercontent.com/wlos/AI_SS/main/14-15-Sequence-Models-and-Transformers/sample_data/support_tickets.csv"
os.makedirs("sample_data", exist_ok=True)
urlretrieve(TICKET_CSV_URL, TICKET_CSV)  # Refresh the course data when rerunning in an existing runtime.
tickets = pd.read_csv(TICKET_CSV)  # Open the supplied labelled data when first used.
print(tickets.groupby(["split", "category"]).size().unstack(fill_value=0))
print(tickets.head(8).to_string(index=False))
assert not tickets["text"].str.lower().duplicated().any(), "Check duplicate ticket text before training."

class_names = ["Account access", "Hardware", "Software", "Network"]
label_to_id = {name: number for number, name in enumerate(class_names)}
id_to_label = {number: name for name, number in label_to_id.items()}
tickets["label"] = tickets["category"].map(label_to_id)
train_tickets = tickets.loc[tickets["split"] == "train"].copy()
validation_tickets = tickets.loc[tickets["split"] == "validation"].copy()
test_tickets = tickets.loc[tickets["split"] == "test"].copy()


**Choose and run the methods**

Complete the MLP, BERT and GPT sections in order. Use the same data splits and label mapping for all three methods. Select models using validation data and keep the test set for the final comparison.


## 2. Feature Engineering + MLP

1. Fit `CountVectorizer` on the training tickets and transform the validation tickets.
2. Train an `MLPClassifier` with 16 hidden units, then another with 32.
3. Retain the model with the higher validation macro F1.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
count_vectorizer = CountVectorizer(lowercase=True, ngram_range=(1, 1))
# Fit the vocabulary on training tickets only; reuse it for validation.
train_counts = count_vectorizer.fit_transform(train_tickets["text"])
validation_counts = count_vectorizer.transform(validation_tickets["text"])
y_count_train = train_tickets["label"].to_numpy()
y_count_validation = validation_tickets["label"].to_numpy()

print("Vocabulary size:", len(count_vectorizer.vocabulary_))
print("Training input shape:", train_counts.shape)
print("First ticket:", train_tickets.iloc[0]["text"])
first_counts = train_counts[0].toarray()[0]  # Densify one row only for inspection.
print([(word, int(first_counts[index])) for index, word in enumerate(count_vectorizer.get_feature_names_out()) if first_counts[index] > 0])


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

COUNT_HIDDEN_UNITS = (16, 32)
count_model = None
best_count_macro_f1 = -1.0
count_validation_rows = []

for hidden_units in COUNT_HIDDEN_UNITS:
    candidate = MLPClassifier(
        hidden_layer_sizes=(hidden_units,), activation="relu", solver="lbfgs",
        alpha=0.01, max_iter=1000, random_state=SEED,
    )
    candidate.fit(train_counts, y_count_train)  # Learn from training tickets only.
    validation_predictions = candidate.predict(validation_counts)
    validation_macro_f1 = f1_score(
        y_count_validation, validation_predictions,
        labels=list(id_to_label), average="macro", zero_division=0,
    )
    count_validation_rows.append({
        "hidden_units": hidden_units,
        "accuracy": accuracy_score(y_count_validation, validation_predictions),
        "macro_f1": validation_macro_f1,
    })
    if validation_macro_f1 > best_count_macro_f1:
        best_count_macro_f1 = validation_macro_f1
        count_model = candidate  # Retain the fitted model selected on validation data.

print(pd.DataFrame(count_validation_rows).to_string(index=False))
print("Selected hidden layer:", count_model.hidden_layer_sizes)


## 3. BERT: Fine-Tuning DistilBERT

1. Convert the shared splits to datasets.
2. Load `distilbert/distilbert-base-uncased` and its tokenizer.
3. Tokenize the tickets with a maximum of 96 tokens and create the four-class model.

The warning about newly initialised classification-head weights is expected; train the model in the following cells.


Run the package installation cell at the start of this notebook first. The training cells print their selected device. Save caches, checkpoints and models under `transformer/` in the current working directory.


In [ ]:
import os
from datasets import Dataset, DatasetDict

ticket_splits = DatasetDict({
    name: Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    for name, frame in [("train", train_tickets), ("validation", validation_tickets), ("test", test_tickets)]
})

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed

BERT_CHECKPOINT = "distilbert/distilbert-base-uncased"
TICKET_TOKENIZER_CACHE_DIR = "transformer/ticket-tokenizer-cache"
os.makedirs(TICKET_TOKENIZER_CACHE_DIR, exist_ok=True)  # Create the local tokenizer cache folder if needed.
# Use the tokenizer paired with the pretrained encoder.
ticket_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT, cache_dir=TICKET_TOKENIZER_CACHE_DIR)
SEED = 42

MAX_TICKET_TOKENS = 96

def prepare_tickets(batch):
    return ticket_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

encoded_tickets = ticket_splits.map(prepare_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Initialise the new task head reproducibly.
TICKET_MODEL_CACHE_DIR = "transformer/ticket-model-cache"
os.makedirs(TICKET_MODEL_CACHE_DIR, exist_ok=True)
ticket_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT, num_labels=len(class_names), cache_dir=TICKET_MODEL_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
)
# Uncomment to freeze the encoder and train only the classification head.
# ticket_model.base_model.requires_grad_(False)
ticket_collator = DataCollatorWithPadding(tokenizer=ticket_tokenizer)


**Train and validate BERT**

Run up to 12 epochs. Evaluate after every epoch, stop after three evaluations without improved validation macro F1, and restore the best validation checkpoint.


In [ ]:
import os
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

def ticket_metrics(prediction):
    logits, labels = prediction
    predicted = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(labels, predicted, labels=list(id_to_label), average="macro", zero_division=0),
    }

TICKET_CHECKPOINT_DIR = "transformer/ticket-checkpoints"
os.makedirs(TICKET_CHECKPOINT_DIR, exist_ok=True)
BERT_MAX_EPOCHS = 12  # Maximum passes; early stopping may finish sooner.
ticket_args = TrainingArguments(
    output_dir=TICKET_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=BERT_MAX_EPOCHS, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
print("BERT training device:", ticket_args.device)  # Show the device selected by Trainer.
ticket_trainer = Trainer(
    model=ticket_model, args=ticket_args,
    train_dataset=encoded_tickets["train"], eval_dataset=encoded_tickets["validation"],
    processing_class=ticket_tokenizer, data_collator=ticket_collator, compute_metrics=ticket_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
ticket_trainer.train()
print("Epochs completed:", ticket_trainer.state.epoch)
print("Best checkpoint:", ticket_trainer.state.best_model_checkpoint)
print("Validation:", ticket_trainer.evaluate())


## 4. GPT: Fine-Tuning GPT-2

1. Load `openai-community/gpt2` and its tokenizer.
2. Set the tokenizer’s padding token and the model’s `pad_token_id` consistently.
3. Tokenize the same ticket splits and create the four-class model.

Keep right padding, the attention mask and `use_cache=False` as shown in the code.


In [ ]:
import os
GPT_CHECKPOINT = "openai-community/gpt2"
GPT_CACHE_DIR = "transformer/gpt-cache"
os.makedirs(GPT_CACHE_DIR, exist_ok=True)
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_CHECKPOINT, cache_dir=GPT_CACHE_DIR)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token  # Reuse an existing token without resizing embeddings.
gpt_tokenizer.padding_side = "right"

def prepare_gpt_tickets(batch):
    return gpt_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

gpt_encoded_tickets = ticket_splits.map(prepare_gpt_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Make the new classification head reproducible.
gpt_model = AutoModelForSequenceClassification.from_pretrained(
    GPT_CHECKPOINT, num_labels=len(class_names), cache_dir=GPT_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
    pad_token_id=gpt_tokenizer.pad_token_id,
)
gpt_model.config.use_cache = False  # Classification does not need a generation cache.
# Uncomment to freeze the decoder and train only the classification head.
# gpt_model.base_model.requires_grad_(False)
gpt_collator = DataCollatorWithPadding(tokenizer=gpt_tokenizer)


**Train and validate GPT**

Reuse the shared metrics. Run up to 12 epochs with a batch size of 4, stop after three evaluations without improved validation macro F1, and restore the best validation checkpoint.


In [ ]:
import os
GPT_CHECKPOINT_DIR = "transformer/gpt-checkpoints"
os.makedirs(GPT_CHECKPOINT_DIR, exist_ok=True)
GPT_MAX_EPOCHS = 12  # Maximum passes; early stopping may finish sooner.
gpt_args = TrainingArguments(
    output_dir=GPT_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    num_train_epochs=GPT_MAX_EPOCHS, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
print("GPT-2 training device:", gpt_args.device)  # Show the device selected by Trainer.
gpt_trainer = Trainer(
    model=gpt_model, args=gpt_args,
    train_dataset=gpt_encoded_tickets["train"], eval_dataset=gpt_encoded_tickets["validation"],
    processing_class=gpt_tokenizer, data_collator=gpt_collator, compute_metrics=ticket_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
gpt_trainer.train()  # Update the decoder and task head using labelled training tickets.
print("Epochs completed:", gpt_trainer.state.epoch)
print("Best checkpoint:", gpt_trainer.state.best_model_checkpoint)
print("GPT-2 validation:", gpt_trainer.evaluate())


## 5. Compare the Three Methods

Evaluate all three methods on the same 64 test tickets. Compare accuracy, macro F1, confusion matrices and misclassified examples with the majority-class baseline. Record the results without using the test set to tune the models.


In [ ]:
import matplotlib.pyplot as plt

after_ticket = ticket_trainer.predict(encoded_tickets["test"])
true_labels = after_ticket.label_ids
gpt_test_output = gpt_trainer.predict(gpt_encoded_tickets["test"])
gpt_predictions = gpt_test_output.predictions.argmax(axis=-1)
assert (gpt_test_output.label_ids == true_labels).all(), "Keep test rows aligned across models."
predicted_labels = after_ticket.predictions.argmax(axis=-1)
test_counts = count_vectorizer.transform(test_tickets["text"])
count_predictions = count_model.predict(test_counts)
majority_label = int(train_tickets["label"].mode().iloc[0])
majority_predictions = [majority_label] * len(true_labels)
comparison_rows = []
for name, predictions in [("Majority class", majority_predictions), ("Feature Engineering + MLP", count_predictions), ("BERT (DistilBERT)", predicted_labels), ("GPT (GPT-2)", gpt_predictions)]:
    comparison_rows.append({
        "model": name, "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, labels=list(id_to_label), average="macro", zero_division=0),
    })
print(pd.DataFrame(comparison_rows).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, name, predictions in [(axes[0], "Feature Engineering + MLP", count_predictions), (axes[1], "BERT (DistilBERT)", predicted_labels), (axes[2], "GPT (GPT-2)", gpt_predictions)]:
    print(name)
    print(classification_report(true_labels, predictions, labels=list(id_to_label), target_names=class_names, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        true_labels, predictions, labels=list(id_to_label), display_labels=class_names,
        xticks_rotation=30, cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()
ticket_results = pd.DataFrame({
    "text": test_tickets["text"].tolist(),
    "actual": [id_to_label[int(label)] for label in true_labels],
    "word_counts_nn": [id_to_label[int(label)] for label in count_predictions],
    "distilbert": [id_to_label[int(label)] for label in predicted_labels],
    "gpt2": [id_to_label[int(label)] for label in gpt_predictions],
})
print("Tickets misclassified by any method:")
prediction_columns = ["word_counts_nn", "distilbert", "gpt2"]
errors = ticket_results[prediction_columns].ne(ticket_results["actual"], axis=0).any(axis=1)
print(ticket_results[errors].to_string(index=False))


## 6. Save, Reload and Route New Tickets

Save the MLP with its vocabulary and label names. Save each Transformer model with its tokenizer, reload the classifiers, and predict categories for the new tickets below.

Inspect ambiguous examples manually. In Colab, download any model files you want to keep before the runtime is deleted.


In [ ]:
import os
import joblib
from transformers import pipeline

COUNT_SAVE_DIR = "transformer/saved-count-mlp"
os.makedirs(COUNT_SAVE_DIR, exist_ok=True)
# Keep the vocabulary, classifier and label names together for consistent inference.
joblib.dump({
    "vectorizer": count_vectorizer, "classifier": count_model, "class_names": class_names,
}, COUNT_SAVE_DIR + "/classifier.joblib")
saved_count = joblib.load(COUNT_SAVE_DIR + "/classifier.joblib")
reloaded_vectorizer = saved_count["vectorizer"]
reloaded_count_model = saved_count["classifier"]

TICKET_SAVE_DIR = "transformer/saved-ticket-classifier"
os.makedirs(TICKET_SAVE_DIR, exist_ok=True)
ticket_trainer.save_model(TICKET_SAVE_DIR)
ticket_tokenizer.save_pretrained(TICKET_SAVE_DIR)
ticket_classifier = pipeline("text-classification", model=TICKET_SAVE_DIR, tokenizer=TICKET_SAVE_DIR)
GPT_SAVE_DIR = "transformer/saved-gpt-classifier"
os.makedirs(GPT_SAVE_DIR, exist_ok=True)
gpt_trainer.save_model(GPT_SAVE_DIR)
gpt_tokenizer.save_pretrained(GPT_SAVE_DIR)
gpt_classifier = pipeline("text-classification", model=GPT_SAVE_DIR, tokenizer=GPT_SAVE_DIR)

new_tickets = [
    "My laptop cannot connect to campus Wi-Fi.",
    "I dropped my monitor and the display is broken.",
    "The program crashes, but only when I connect to Wi-Fi.",
]
new_counts = reloaded_vectorizer.transform(new_tickets)
count_classes = reloaded_count_model.predict(new_counts)
bert_results = ticket_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)
gpt_results = gpt_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)
for text, count_class, bert_result, gpt_result in zip(new_tickets, count_classes, bert_results, gpt_results):
    print(text)
    print("Feature Engineering + MLP:", saved_count["class_names"][count_class])
    print("BERT (DistilBERT):", bert_result, "| GPT (GPT-2):", gpt_result)
